In [3]:
import json
import pandas as pd
import spacy
from collections import Counter
import os
os.chdir(r"E:\EBM-NLP") 

### Load the Training Dataset

In [4]:
with open("cleaned_data/train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

print(len(train_data))
print(train_data[0].keys())

4742
dict_keys(['pmid', 'split', 'text', 'tokens', 'labels', 'spans'])


### Check the .json file

In [5]:
print("PMID:", train_data[0]["pmid"])
print("Split:", train_data[0]["split"])
print("Text preview:\n", train_data[0]["text"])
print("Tokens:", train_data[0]["tokens"])
print("Labels sample:", train_data[0]["labels"])
print("Spans sample:", train_data[0]["spans"])

PMID: 24679062
Split: train
Text preview:
 Aspirin in patients undergoing noncardiac surgery. BACKGROUND There is substantial variability in the perioperative administration of aspirin in patients undergoing noncardiac surgery, both among patients who are already on an aspirin regimen and among those who are not. METHODS Using a 2-by-2 factorial trial design, we randomly assigned 10,010 patients who were preparing to undergo noncardiac surgery and were at risk for vascular complications to receive aspirin or placebo and clonidine or placebo. The results of the aspirin trial are reported here. The patients were stratified according to whether they had not been taking aspirin before the study (initiation stratum, with 5628 patients) or they were already on an aspirin regimen (continuation stratum, with 4382 patients). Patients started taking aspirin (at a dose of 200 mg) or placebo just before surgery and continued it daily (at a dose of 100 mg) for 30 days in the initiation stratum and 

 Find triggers/keywords

 Start by filtering each token into PIO lists

In [6]:
P_tokens = []
I_tokens = []
O_tokens = []

for doc in train_data:
    tokens = doc["tokens"]
    labels = doc["labels"]

    P = labels.get("participants", [])
    I = labels.get("interventions", labels.get("intervention", []))
    O = labels.get("outcomes", [])

    P_tokens.extend([t.lower() for t, l in zip(tokens, P) if l == 1])
    I_tokens.extend([t.lower() for t, l in zip(tokens, I) if l == 1])
    O_tokens.extend([t.lower() for t, l in zip(tokens, O) if l == 1])

### Clean the tokens

In [7]:
nlp = spacy.load("en_core_web_sm")
stopwords = nlp.Defaults.stop_words

def clean_tokens(tokens):
    cleaned = []
    
    for t in tokens:
        t = t.lower()
        
        if t in stopwords:
            continue
        if not t.isalpha():   
            continue
        
        cleaned.append(t)
    
    return cleaned

P_tokens_clean = clean_tokens(P_tokens)
I_tokens_clean = clean_tokens(I_tokens)
O_tokens_clean = clean_tokens(O_tokens)

 Check frequency of cleaned tokens

In [8]:
P_freq = Counter(P_tokens_clean)
I_freq = Counter(I_tokens_clean)
O_freq = Counter(O_tokens_clean)

print("Top P tokens:", P_freq.most_common(50))
print("Top I tokens:", I_freq.most_common(50))
print("Top O tokens:", O_freq.most_common(50))

Top P tokens: [('patients', 6517), ('children', 1697), ('cancer', 955), ('group', 935), ('years', 906), ('women', 901), ('autism', 820), ('subjects', 734), ('study', 582), ('age', 571), ('n', 513), ('healthy', 484), ('disease', 480), ('treatment', 468), ('randomized', 428), ('chronic', 424), ('undergoing', 405), ('acute', 383), ('disorder', 374), ('surgery', 373), ('spectrum', 364), ('participants', 359), ('adults', 349), ('men', 343), ('trial', 332), ('asd', 321), ('aged', 315), ('total', 313), ('disorders', 291), ('mean', 290), ('treated', 288), ('control', 286), ('breast', 283), ('enrolled', 276), ('groups', 269), ('randomly', 263), ('volunteers', 260), ('autistic', 250), ('heart', 247), ('advanced', 242), ('received', 228), ('coronary', 228), ('syndrome', 226), ('type', 221), ('male', 215), ('primary', 211), ('pain', 210), ('severe', 208), ('young', 208), ('months', 207)]
Top I tokens: [('placebo', 1878), ('therapy', 1025), ('group', 810), ('intervention', 651), ('treatment', 639),

Using the frequency of tokens in the code block above, we can identify the following cues. 

From the frequency list for PIO tokens, we selected words that refer directly to the patients, interventions, and outcomes.

Certain words, that may be frequent in one token list may be omitted due to having overlap with another. (e.g., group, or blood)

In [9]:
P_cues = {
    "patients", "children", "women", "subjects", "participants", "adults", "men", "volunteers"
}

I_cues = {
    "placebo", "therapy", "intervention", "treatment", "received", "receive", "randomized",
    "plus", "versus", "control", "standard", "conventional", "combined", "chemotherapy", "radiotherapy", 
    "surgery", "exercise", "training", "diet", "program", "combination", "oral", "intravenous", "infusion", 
    "supplementation", "vaccine", "vaccination", "dose", "daily"
}

O_cues = {
    "rate", "pain", "levels", "survival", "pressure", "efficacy", "response", "effects", "scores", "rates",
    "symptoms", "adverse", "scale", "safety", "function", "score", "quality", "effect", "life", "duration", 
    "risk", "level", "changes", "mortality"
}

### Final Cues after Hand Crafting any Additional Cues

In [10]:
P_cues = {
    'patients', 'subjects', 'participants', 'children', 'adults', 'women',
    'men', 'volunteers', 'individuals', 'infants', 'adolescents', 'elderly',
    'enrolled', 'recruited', 'randomized', 'randomised', 'aged', 'diagnosed',
    'healthy', 'male', 'female', 'obese', 'pregnant', 'undergoing',
    'consecutive', 'outpatients', 'inpatients', 'population', 'cohort',
    'screened', 'eligible', 'included', 'excluded', 'males', 'females',
    'newborns', 'neonates', 'preschool', 'postmenopausal'
}

I_cues = {
    'treatment', 'therapy', 'drug', 'dose', 'placebo', 'intervention',
    'administered', 'received', 'mg', 'daily', 'twice', 'oral', 'topical',
    'intravenous', 'injection', 'surgery', 'versus', 'compared', 'plus',
    'combination', 'regimen', 'assigned', 'allocated', 'saline', 'control',
    'capsule', 'tablet', 'infusion', 'subcutaneous', 'extract', 'supplement',
    'vaccine', 'antibiotic', 'steroid', 'inhibitor', 'agonist', 'antagonist',
    'blocker', 'counseling', 'counselling', 'program', 'programme',
    'exercise', 'training', 'acupuncture', 'massage', 'physiotherapy'
}

O_cues = {
    'outcome', 'efficacy', 'safety', 'response', 'survival', 'mortality',
    'improvement', 'reduction', 'score', 'rate', 'adverse', 'events',
    'pain', 'quality', 'function', 'levels', 'change', 'difference',
    'endpoint', 'measure', 'tolerability', 'remission', 'relapse', 'death',
    'recurrence', 'complication', 'satisfaction', 'duration', 'incidence',
    'prevalence', 'symptom', 'symptoms', 'cure', 'healing', 'recovery',
    'biomarker', 'concentration', 'clearance', 'eradication', 'morbidity',
    'hospitalization', 'readmission', 'infection'
}